# Калибровка вероятностей, кривые принятия решений и выбор порога

Финалисты под сервис и интерпретацию - четыре семейства (логистическая регрессия, случайный лес, XGBoost, CatBoost) на двух наборах признаков (significant 10 и no_collinear 25), восемь моделей. Гиперпараметры взяты из ноутбука 07 (Optuna). Здесь решаем три задачи: проверяем, насколько вероятности моделей откалиброваны и нужна ли калибровка, смотрим клиническую пользу через кривые принятия решений, и выбираем рабочий порог для каждого финалиста.

Все настройки делаем на train по out-of-fold предсказаниям (5-фолдовая стратифицированная CV, `cross_val_predict`), калибраторы обучаются внутри фолдов - утечки нет. Отложенный тест трогаем только в самом конце, зафиксированным порогом.

## Калибровка: теория

Калибровка о том, совпадает ли предсказанная вероятность с наблюдаемой частотой. Если модель присвоила группе пациентов риск 0.3, то среди них рецидив должен случиться примерно у 30%. Это отдельное свойство от разделяющей способности: ROC-AUC меряет только то, как модель ранжирует пациентов (риск больного выше риска здорового), но ничего не говорит про абсолютную верность чисел. Модель с высоким ROC-AUC может систематически завышать или занижать риск.

Для сервиса поддержки решений это важно напрямую: врач смотрит на саму вероятность, а не на бинарную метку. Завышенный риск ведет к лишним вмешательствам, заниженный - к пропуску профилактики.

Как оцениваем калибровку:

- надежностная кривая (reliability diagram): пациентов делим на бины по предсказанной вероятности, по оси x откладываем среднюю предсказанную вероятность в бине, по оси y - наблюдаемую долю рецидивов. Идеальная калибровка лежит на диагонали. Точки выше диагонали - модель занижает риск, ниже - завышает.
- Brier score: средний квадрат разницы между вероятностью и исходом (0 или 1). Ниже - лучше. Brier объединяет калибровку и разрешающую способность в одно число, поэтому удобен для сравнения вариантов.

Способы калибровки:

- Платт (сигмоида): поверх вероятностей модели подгоняется логистическая кривая с двумя параметрами. Устойчив на малых выборках, но предполагает, что искажение имеет сигмоидную форму.
- изотоническая регрессия: монотонная ступенчатая подгонка, гибче сигмоиды и ловит произвольную монотонную деформацию, но требует больше данных и на малых выборках переобучается.

Важное свойство: и Платт, и изотоническая преобразуют вероятности монотонно, то есть не меняют порядок пациентов по риску. Поэтому ROC-AUC после калибровки не меняется. Меняются Brier, абсолютные вероятности и численное значение порога, но достижимая пара чувствительность-специфичность остается прежней. Это нам пригодится: выбор рабочей точки по чувствительности от калибровки не зависит, калибровка нужна, чтобы сама вероятность для врача была честной.

Калибровку оцениваем по out-of-fold на train (калибратор обучается внутри фолдов), а не на тесте, чтобы не подсматривать. Решение калибровать или нет принимаем для каждого финалиста по OOF Brier и виду кривой.

In [ ]:
import sys, pathlib, json, warnings
p = pathlib.Path.cwd()
while not (p / 'src').exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss, roc_auc_score

from src import io, features, config, threshold as thr, evaluation
from src import optuna_tuning as ot
from src.viz import apply_style
%matplotlib inline
apply_style()

df = io.load_processed()
X_train, X_test, y_train, y_test = features.make_split(df)
y_tr = y_train.to_numpy()
params_all = json.loads(
    (config.TABLES_DIR / 'tuning_optuna_params.json').read_text(encoding='utf-8'))

MODELS = ['logreg', 'rf', 'xgb', 'catboost']
SETS = ['significant', 'no_collinear']
FINALISTS = [(m, fs) for fs in SETS for m in MODELS]
skf = StratifiedKFold(5, shuffle=True, random_state=config.RANDOM_SEED)


def build(model, fset):
    feats = features.feature_sets(df)[fset]
    pipe = ot._build(model, feats, df, y_train, params_all[f'{model}|{fset}'])
    return pipe, feats


print('Финалисты:', FINALISTS)
print(f'train {len(y_train)} (рецидивов {int(y_tr.sum())}), test {len(y_test)}')

In [ ]:
# Сырые out-of-fold вероятности на train для каждого финалиста (без утечки).
# Используем и для калибровки, и для выбора порога.
oof = {}
for model, fset in FINALISTS:
    pipe, feats = build(model, fset)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        proba = cross_val_predict(pipe, X_train[feats], y_train, cv=skf,
                                  method='predict_proba')[:, 1]
    oof[(model, fset)] = proba
    print(f'{model:9} {fset:13} OOF ROC-AUC={roc_auc_score(y_tr, proba):.3f}')

### Калибровка финалистов: Brier и надежностные кривые

Для каждого финалиста сравниваем сырые вероятности с калиброванными по Платту и изотонической, все по out-of-fold на train. Калибратор обучается внутри фолдов внешней CV (вложенная схема), поэтому оценка честная.

In [ ]:
calib_oof = {}    # (model, fset) -> {вариант: proba}
calib_rows = []
for model, fset in FINALISTS:
    pipe, feats = build(model, fset)
    variants = {'сырые': oof[(model, fset)]}
    for method, label in [('sigmoid', 'Платт'), ('isotonic', 'изотон.')]:
        cc = CalibratedClassifierCV(pipe, method=method, cv=5)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            variants[label] = cross_val_predict(cc, X_train[feats], y_train, cv=skf,
                                                method='predict_proba')[:, 1]
    calib_oof[(model, fset)] = variants
    for label, proba in variants.items():
        calib_rows.append({'модель': model, 'набор': fset, 'вариант': label,
                           'Brier': round(brier_score_loss(y_tr, proba), 3),
                           'ROC-AUC': round(roc_auc_score(y_tr, proba), 3)})
    print(f'{model} | {fset} готов')

calib = pd.DataFrame(calib_rows)
calib.to_csv(config.TABLES_DIR / 'calibration_oof.csv', index=False,
             encoding='utf-8-sig')
# Сводка Brier по вариантам (строки - финалисты, столбцы - вариант калибровки).
display(calib.pivot_table(index=['набор', 'модель'], columns='вариант',
                          values='Brier').round(3))

In [ ]:
# Надежностные кривые: сетка 2x4, по строке - набор признаков, по столбцу - модель.
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, (model, fset) in zip(axes.ravel(), FINALISTS):
    ax.plot([0, 1], [0, 1], 'k:', lw=1, label='идеал')
    for label, proba in calib_oof[(model, fset)].items():
        frac, mean = calibration_curve(y_tr, proba, n_bins=5, strategy='quantile')
        ax.plot(mean, frac, 'o-', ms=4, label=f'{label} (Brier {brier_score_loss(y_tr, proba):.3f})')
    ax.set_title(f'{model}, {fset}', fontsize=10)
    ax.set_xlabel('предсказанная вероятность')
    ax.set_ylabel('наблюдаемая доля')
    ax.legend(fontsize=7)
fig.suptitle('Надежностные кривые (out-of-fold на train)', fontsize=13)
fig.tight_layout()
config.ensure_dirs()
fig.savefig(config.FIGURES_DIR / 'calibration_finalists.png', bbox_inches='tight')
plt.show()

## Кривые принятия решений (DCA)

Кривая принятия решений показывает чистую пользу модели в зависимости от пороговой вероятности. Чистая польза = TP/n - (FP/n) * pt/(1-pt), где пороговая вероятность pt задает обменный курс между ложной тревогой и пропуском: при пороге pt врач считает, что один пропущенный рецидив стоит как (1-pt)/pt ложных тревог. Модель сравнивается с двумя крайними стратегиями: лечить всех (помечать всех как риск) и не лечить никого. Модель полезна в том диапазоне порогов, где ее кривая выше обеих крайних.

Строим по out-of-fold на train (сырые вероятности; калибровка монотонна и кривую достижимой пользы не меняет).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, (model, fset) in zip(axes.ravel(), FINALISTS):
    thr_grid, nb_model, nb_all = evaluation.decision_curve(y_tr, oof[(model, fset)])
    ax.plot(thr_grid, nb_model, color='#C44E52', lw=2, label='модель')
    ax.plot(thr_grid, nb_all, color='#4C72B0', ls='--', label='лечить всех')
    ax.axhline(0, color='#999', ls=':', label='не лечить никого')
    ax.set_title(f'{model}, {fset}', fontsize=10)
    ax.set_xlabel('пороговая вероятность')
    ax.set_ylabel('чистая польза')
    ax.set_ylim(-0.05, max(nb_model) * 1.2 + 0.01)
    ax.legend(fontsize=7)
fig.suptitle('Кривые принятия решений (out-of-fold на train)', fontsize=13)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / 'dca_finalists.png', bbox_inches='tight')
plt.show()

## Выбор порога

Порог подбираем на train по out-of-fold вероятностям, для каждого финалиста свой (вероятности у моделей разной шкалы). Основной критерий - фиксированная целевая чувствительность с максимальной специфичностью: сначала ставим планку по доле отлавливаемых рецидивов, затем берем самый строгий порог, который ее держит. Для сравнения показываем индекс Юдена (сбалансированная точка) и F2 (вес в сторону полноты).

Выборка мала (в train 69 рецидивов, шаг по чувствительности грубый), поэтому цель смотрим в зоне 0.75-0.80. Ниже - кривые чувствительность-специфичность по порогу и таблица: какой порог и какая специфичность выходят при цели 0.75 и при цели 0.80. По ним выбираем точное значение цели.

In [ ]:
# Кривые чувствительности и специфичности по порогу для каждого финалиста.
grid = np.linspace(0.02, 0.98, 193)
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, (model, fset) in zip(axes.ravel(), FINALISTS):
    proba = oof[(model, fset)]
    sens = [thr.metrics_at(y_tr, proba, t)['чувств.'] for t in grid]
    spec = [thr.metrics_at(y_tr, proba, t)['специф.'] for t in grid]
    ax.plot(grid, sens, color='#C44E52', label='чувствительность')
    ax.plot(grid, spec, color='#4C72B0', label='специфичность')
    for st, c in [(0.75, '#55A868'), (0.80, '#8172B3')]:
        ch = thr.select_thresholds(y_tr, proba, sens_target=st)[f'чувств.>={st}']
        ax.axvline(ch, ls='--', color=c, lw=1, label=f'sens>={st} (порог {ch:.2f})')
    ax.set_title(f'{model}, {fset}', fontsize=10)
    ax.set_xlabel('порог вероятности')
    ax.set_ylabel('метрика')
    ax.legend(fontsize=7)
fig.suptitle('Чувствительность и специфичность по порогу (out-of-fold на train)', fontsize=13)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / 'threshold_finalists.png', bbox_inches='tight')
plt.show()

In [ ]:
# Таблица: порог и метрики при цели чувствительности 0.75 и 0.80, плюс Юден, на train OOF.
rows = []
for model, fset in FINALISTS:
    proba = oof[(model, fset)]
    ch = thr.select_thresholds(y_tr, proba, sens_target=0.75)
    ch80 = thr.select_thresholds(y_tr, proba, sens_target=0.80)
    for label, t in [('Юден', ch['Youden']),
                     ('sens>=0.75', ch['чувств.>=0.75']),
                     ('sens>=0.80', ch80['чувств.>=0.8'])]:
        rows.append({'модель': model, 'набор': fset, 'критерий': label,
                     **thr.metrics_at(y_tr, proba, t)})
thr_train = pd.DataFrame(rows)
thr_train.to_csv(config.TABLES_DIR / 'threshold_train_oof.csv', index=False,
                 encoding='utf-8-sig')
display(thr_train)

## Калибровка и кривые: вывод

Калибровку применяем выборочно, там где она реально снижает Brier. Правило простое: для финалиста берем Платта, только если он улучшает OOF Brier не меньше чем на 0.005, иначе оставляем сырые вероятности. Порог в 0.005 отсекает шумовые колебания на малой выборке.

По этому правилу Платт включается у двух моделей на слабом наборе significant: logreg (0.199 против 0.189) и rf (0.203 против 0.183), там сырые вероятности были заметно хуже. У всех остальных финалистов, в том числе у всех на основном наборе no_collinear, сырые вероятности уже хорошо откалиброваны (Brier 0.178-0.189), и Платт их не улучшает или слегка ухудшает. Там сырые вероятности валиднее: они не несут лишней дисперсии от подгонки калибратора на 218 наблюдениях. Изотоническую не разворачиваем нигде, на такой выборке она переобучается.

Отдельно про ROC-AUC: его снижение при калибровке в таблице выше (например rf на no_collinear с 0.772 до 0.744) - артефакт вложенной переподгонки калибратора на урезанных данных, а не реальная потеря. Калибровка монотонна и ранжирование не меняет.

Кривые принятия решений: модели дают чистую пользу выше стратегий лечить всех и не лечить никого в клинически разумном диапазоне порогов, что подтверждает их практическую ценность.

## Фиксация порога и проверка на тесте

Порог фиксируем по целевой чувствительности 0.75 на train по out-of-fold вероятностям, для каждого финалиста свой и на тех вероятностях, что идут в развертывание (калиброванных, если выбран Платт, иначе сырых). Затем применяем зафиксированный порог к отложенному тесту - это единственное обращение к тесту в ноутбуке. Конфигурацию финалистов (набор признаков, режим калибровки, порог) сохраняем в `finalists_config.json` для сервиса и дальнейших этапов.

In [ ]:
# Выбор калибровки по финалистам: Платт, только если он снижает OOF Brier >=0.005,
# иначе сырые вероятности (они валиднее, без лишней дисперсии). Порог по целевой
# чувствительности 0.75 фиксируем на тех вероятностях, что идут в развертывание.
SENS_TARGET = 0.75
BRIER_MARGIN = 0.005
y_te = y_test.to_numpy()
rows = []
finalists_cfg = {}
for model, fset in FINALISTS:
    pipe, feats = build(model, fset)
    b_raw = brier_score_loss(y_tr, calib_oof[(model, fset)]['сырые'])
    b_platt = brier_score_loss(y_tr, calib_oof[(model, fset)]['Платт'])
    use_platt = b_platt <= b_raw - BRIER_MARGIN

    # Вероятности для подбора порога - те же, что развернем (калиброванные или сырые).
    oof_use = calib_oof[(model, fset)]['Платт'] if use_platt else oof[(model, fset)]
    t = thr.select_thresholds(y_tr, oof_use, sens_target=SENS_TARGET)[f'чувств.>={SENS_TARGET}']

    est = CalibratedClassifierCV(pipe, method='sigmoid', cv=5) if use_platt else pipe
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        est.fit(X_train[feats], y_train)
        proba_test = est.predict_proba(X_test[feats])[:, 1]

    m_tr = thr.metrics_at(y_tr, oof_use, t)
    m_te = thr.metrics_at(y_te, proba_test, t)
    rows.append({'модель': model, 'набор': fset,
                 'калибровка': 'Платт' if use_platt else 'сырые',
                 'порог': round(float(t), 3),
                 'sens train': m_tr['чувств.'], 'spec train': m_tr['специф.'],
                 'sens test': m_te['чувств.'], 'spec test': m_te['специф.'],
                 'PPV test': m_te['PPV'], 'NPV test': m_te['NPV']})
    finalists_cfg[f'{model}|{fset}'] = {
        'feature_set': fset,
        'calibration': 'sigmoid' if use_platt else 'none',
        'threshold': round(float(t), 4),
        'sens_target': SENS_TARGET}

test_thr = pd.DataFrame(rows)
test_thr.to_csv(config.TABLES_DIR / 'threshold_test.csv', index=False, encoding='utf-8-sig')
(config.TABLES_DIR / 'finalists_config.json').write_text(
    json.dumps(finalists_cfg, ensure_ascii=False, indent=2), encoding='utf-8')
display(test_thr)

## Вывод

Калибровка включилась по правилу ровно там, где Brier этого требовал: Платт у logreg и rf на наборе significant, сырые вероятности у остальных шести финалистов. Режим калибровки и порог каждого финалиста сохранены в `finalists_config.json`.

Порог под целевую чувствительность 0.75 на train держится у всех финалистов (0.754-0.783). На отложенном тесте порог фиксированный, поэтому чувствительность колеблется вокруг цели: при 18 рецидивах в тесте шаг по чувствительности около 5.6%, и точное попадание невозможно по построению. На тесте чувствительность легла в диапазон 0.50-0.89, специфичность 0.49-0.73.

Лучшие рабочие точки на тесте у XGBoost: на significant чувствительность 0.72 при специфичности 0.73 (PPV 0.57, NPV 0.84), на no_collinear 0.67 и 0.73. Высокую чувствительность 0.89 при специфичности 0.62 дали rf на significant и logreg на no_collinear - они почти не пропускают рецидивы ценой большего числа ложных тревог. Слабее всех на тесте оказался CatBoost на no_collinear: чувствительность упала до 0.50, это в пределах разброса малого теста, но в сервисе такую неустойчивость надо учитывать.

Что это значит для сервиса. Восемь финалистов с зафиксированными порогами и режимом калибровки готовы к развертыванию. Высокий NPV (0.75-0.92) говорит, что отрицательному ответу модели можно доверять - это полезно, чтобы выделять группу низкого риска. PPV умеренный (0.41-0.73) при распространенности рецидива около 32%, поэтому положительный ответ требует клинического подтверждения. Окончательный отбор моделей для интерфейса делаем после стекинга и интерпретации.